# Barbell trajectory ? phase label alignment probe

Purpose: visually check whether YOLO-World barbell trajectories line up with the phase labels (`start`, `bottom`, `finish`).

What to look for:
- If the barbell turning point consistently appears near each red `bottom` line, the label/signal alignment is plausible.
- If the barbell turning point is shifted, missing, flat, or noisy around the red line, phase performance is probably limited by label/signal alignment or detector trajectory quality rather than model capacity.

The notebook is read-only by default. It only writes optional PNG plots under `phase_experiments/barbell_yolo_world/label_alignment_probe/` when `SAVE_PLOTS=True`.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Import repo parsing/feature helpers so this notebook uses the same label and cache contract as training.
from model import train_ablation as ta

PROJECT_ROOT = Path.cwd()

# Robust data-root detection: the dataset directory name may display differently across consoles/codepages.
DATA_ROOT_CANDIDATES = [PROJECT_ROOT / "data" / "??" / "data"]
DATA_ROOT_CANDIDATES += sorted((PROJECT_ROOT / "data").glob("*/data"))
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Could not find data root under {PROJECT_ROOT / 'data'}")

LABEL_DIR = DATA_ROOT / "labels"
BARBELL_DIR = DATA_ROOT / "STGCN_LSTM" / "barbell_yolo_world"
VIDEO_DIR = DATA_ROOT
OUT_DIR = PROJECT_ROOT / "phase_experiments" / "barbell_yolo_world" / "label_alignment_probe"

# Display/filter controls.
EXERCISE = None            # None, "squat", "benchpress", or "deadlift"
NAME_CONTAINS = None       # e.g. "stu7" or "bench press_1"; None disables
LIMIT = 12                 # number of videos to plot in batch cell
SORT_BY = "type_name"      # "type_name", "detection_rate", "mean_conf", "n_reps", "frames"
SAVE_PLOTS = True
INVERT_Y_AXIS = True       # True makes visually higher barbell positions appear higher on the plot
SHOW_RAW_CENTER = True
SHOW_CONFIDENCE = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("LABEL_DIR:", LABEL_DIR)
print("BARBELL_DIR:", BARBELL_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def load_records() -> pd.DataFrame:
    labels = ta.load_labels(LABEL_DIR)
    rows: List[Dict[str, Any]] = []
    for (typ, name), info in labels.items():
        npz_path = ta.pose_path(
            typ,
            name,
            pose_backend=ta.POSE_BACKEND_BARBELL,
            barbell_dir=BARBELL_DIR,
        )
        video_path = VIDEO_DIR / typ / name
        if not npz_path.exists():
            continue
        with np.load(npz_path, allow_pickle=True) as d:
            kpts = d["kpts"]
            detected = d["detected"] if "detected" in d.files else (kpts[:, 0, 2] > 0)
            conf = d["detection_conf"] if "detection_conf" in d.files else kpts[:, 0, 2]
            rows.append(
                {
                    "type": typ,
                    "name": name,
                    "video_path": str(video_path),
                    "npz": str(npz_path),
                    "frames": int(d["num_frames"]) if "num_frames" in d.files else int(kpts.shape[0]),
                    "fps": float(d["fps"]) if "fps" in d.files else 30.0,
                    "n_reps": len(info["reps"]),
                    "reps": info["reps"],
                    "incomplete": info.get("incomplete"),
                    "detection_rate": float(np.asarray(detected, dtype=bool).mean()) if len(detected) else 0.0,
                    "mean_conf": float(np.asarray(conf, dtype=np.float32)[np.asarray(detected, dtype=bool)].mean()) if np.asarray(detected, dtype=bool).any() else 0.0,
                }
            )
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No barbell cache records found. Run model/extract_barbell_yolo_world.py first.")
    if SORT_BY == "type_name":
        df = df.sort_values(["type", "name"]).reset_index(drop=True)
    elif SORT_BY in df.columns:
        df = df.sort_values(SORT_BY, ascending=True).reset_index(drop=True)
    return df

records = load_records()
print("records:", len(records))
display(records.groupby("type").agg(videos=("name", "count"), frames=("frames", "sum"), detection_rate=("detection_rate", "mean"), mean_conf=("mean_conf", "mean")))
display(records.head(10)[["type", "name", "frames", "fps", "n_reps", "detection_rate", "mean_conf", "npz"]])

In [ ]:
def load_trace(npz_path: str | Path) -> Dict[str, np.ndarray]:
    """Return raw/smoothed barbell y traces from one extracted cache."""
    with np.load(npz_path, allow_pickle=True) as d:
        kpts = d["kpts"].astype(np.float32)
        smooth_y = kpts[:, 0, 1]
        smooth_x = kpts[:, 0, 0]
        conf = d["detection_conf"].astype(np.float32) if "detection_conf" in d.files else kpts[:, 0, 2].astype(np.float32)
        detected = d["detected"].astype(bool) if "detected" in d.files else (conf > 0)
        boxes = d["boxes"].astype(np.float32) if "boxes" in d.files else None
        if boxes is not None and len(boxes):
            raw_y = (boxes[:, 1] + boxes[:, 3]) / 2.0
            raw_x = (boxes[:, 0] + boxes[:, 2]) / 2.0
            raw_y[~np.isfinite(raw_y)] = np.nan
            raw_x[~np.isfinite(raw_x)] = np.nan
        else:
            raw_y = smooth_y.copy()
            raw_x = smooth_x.copy()
        return {
            "smooth_x": smooth_x,
            "smooth_y": smooth_y,
            "raw_x": raw_x,
            "raw_y": raw_y,
            "conf": conf,
            "detected": detected,
        }


def rep_turning_summary(y: np.ndarray, reps: Iterable[Tuple[int, int, int]], search_pad: int = 8) -> pd.DataFrame:
    """For each rep, report nearest local min/max offsets around the labeled bottom frame.

    This intentionally reports both min and max because image-coordinate direction can vary by exercise/camera.
    """
    rows = []
    T = len(y)
    for rep_idx, (s, b, f) in enumerate(reps, start=1):
        s = int(np.clip(s, 0, max(T - 1, 0)))
        b = int(np.clip(b, 0, max(T - 1, 0)))
        f = int(np.clip(f, 0, max(T - 1, 0)))
        lo = max(0, s - search_pad)
        hi = min(T, f + search_pad + 1)
        seg = y[lo:hi]
        finite = np.isfinite(seg)
        if len(seg) == 0 or not finite.any():
            rows.append({"rep": rep_idx, "start": s, "bottom": b, "finish": f, "max_frame": np.nan, "max_offset": np.nan, "min_frame": np.nan, "min_offset": np.nan})
            continue
        safe = seg.copy()
        safe[~finite] = np.nan
        max_frame = int(lo + np.nanargmax(safe))
        min_frame = int(lo + np.nanargmin(safe))
        rows.append(
            {
                "rep": rep_idx,
                "start": s,
                "bottom": b,
                "finish": f,
                "max_frame": max_frame,
                "max_offset": max_frame - b,
                "min_frame": min_frame,
                "min_offset": min_frame - b,
                "y_start": float(y[s]) if np.isfinite(y[s]) else np.nan,
                "y_bottom": float(y[b]) if np.isfinite(y[b]) else np.nan,
                "y_finish": float(y[f]) if np.isfinite(y[f]) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def plot_video(row: pd.Series, *, save: bool = SAVE_PLOTS, show: bool = True) -> pd.DataFrame:
    trace = load_trace(row["npz"])
    y = trace["smooth_y"]
    raw_y = trace["raw_y"]
    conf = trace["conf"]
    detected = trace["detected"]
    T = len(y)
    frames = np.arange(T)
    reps = list(row["reps"])

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(frames, y, label="smoothed center-y from kpts", color="tab:blue", linewidth=1.8)
    if SHOW_RAW_CENTER:
        ax.scatter(frames, raw_y, label="raw bbox center-y", color="black", s=7, alpha=0.25)
    miss = ~detected
    if miss.any():
        ax.scatter(frames[miss], y[miss], label="missing/interpolated", color="red", s=16, alpha=0.6)

    for rep_idx, (s, b, f) in enumerate(reps, start=1):
        s, b, f = int(s), int(b), int(f)
        ax.axvspan(s, b, color="orange", alpha=0.10)
        ax.axvspan(b, f, color="royalblue", alpha=0.08)
        ax.axvline(s, color="gray", linestyle="--", linewidth=1.0)
        ax.axvline(b, color="red", linestyle="-", linewidth=1.4)
        ax.axvline(f, color="green", linestyle="--", linewidth=1.0)
        y_text = np.nanmin(y) if np.isfinite(y).any() else 0.0
        ax.text(b, y_text, f"B{rep_idx}", color="red", fontsize=8, rotation=90, va="bottom", ha="right")

    ax.set_title(
        f"{row['type']} / {row['name']} | reps={len(reps)} | frames={T} | "
        f"det_rate={row['detection_rate']:.3f} | mean_conf={row['mean_conf']:.3f}"
    )
    ax.set_xlabel("frame")
    ax.set_ylabel("normalized barbell center-y (0=top, 1=bottom)")
    ax.grid(alpha=0.25)
    if INVERT_Y_AXIS:
        ax.invert_yaxis()

    if SHOW_CONFIDENCE:
        ax2 = ax.twinx()
        ax2.plot(frames, conf, label="confidence", color="tab:purple", alpha=0.35, linewidth=1.0)
        ax2.set_ylabel("YOLO-World confidence")
        ax2.set_ylim(0, max(1.0, float(np.nanmax(conf)) if len(conf) else 1.0))
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
    else:
        ax.legend(loc="upper right")

    plt.tight_layout()
    if save:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        safe_name = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in f"{row['type']}_{row['name']}")
        fig.savefig(OUT_DIR / f"{safe_name}.png", dpi=160)
    if show:
        plt.show()
    else:
        plt.close(fig)

    return rep_turning_summary(y, reps)

In [ ]:
# Batch plot according to controls at the top.
subset = records.copy()
if EXERCISE:
    subset = subset[subset["type"].eq(EXERCISE)]
if NAME_CONTAINS:
    subset = subset[subset["name"].str.contains(NAME_CONTAINS, case=False, regex=False)]

if SORT_BY == "type_name":
    subset = subset.sort_values(["type", "name"])
elif SORT_BY in subset.columns:
    subset = subset.sort_values(SORT_BY, ascending=True)

print(f"Plotting {min(LIMIT, len(subset))} / {len(subset)} selected videos")
all_turns = []
for _, row in subset.head(LIMIT).iterrows():
    turn_df = plot_video(row, save=SAVE_PLOTS, show=True)
    turn_df.insert(0, "name", row["name"])
    turn_df.insert(0, "type", row["type"])
    all_turns.append(turn_df)

turns = pd.concat(all_turns, ignore_index=True) if all_turns else pd.DataFrame()
display(turns)
if SAVE_PLOTS:
    print("Saved plots to:", OUT_DIR)

In [ ]:
# Plot lowest-confidence / worst-detection examples first.
worst = records.sort_values(["detection_rate", "mean_conf"], ascending=[True, True]).head(LIMIT)
display(worst[["type", "name", "frames", "n_reps", "detection_rate", "mean_conf", "npz"]])

all_turns = []
for _, row in worst.iterrows():
    turn_df = plot_video(row, save=SAVE_PLOTS, show=True)
    turn_df.insert(0, "name", row["name"])
    turn_df.insert(0, "type", row["type"])
    all_turns.append(turn_df)

worst_turns = pd.concat(all_turns, ignore_index=True) if all_turns else pd.DataFrame()
display(worst_turns)

In [ ]:
# Manually inspect one video by type/name.
# Example:
# SELECT_TYPE = "squat"
# SELECT_NAME = "squat_2.MOV"
SELECT_TYPE = None
SELECT_NAME = None

if SELECT_TYPE and SELECT_NAME:
    hit = records[(records["type"] == SELECT_TYPE) & (records["name"] == SELECT_NAME)]
    if hit.empty:
        raise ValueError(f"No record found for {SELECT_TYPE}/{SELECT_NAME}")
    turn_df = plot_video(hit.iloc[0], save=SAVE_PLOTS, show=True)
    display(turn_df)
else:
    print("Set SELECT_TYPE and SELECT_NAME, then re-run this cell.")

In [ ]:
# Export a compact per-rep turning-point offset table for all videos without showing plots.
# Negative offset means the local extremum occurs before the labeled bottom; positive means after.
rows = []
for _, row in records.iterrows():
    trace = load_trace(row["npz"])
    ts = rep_turning_summary(trace["smooth_y"], row["reps"])
    if not ts.empty:
        ts.insert(0, "name", row["name"])
        ts.insert(0, "type", row["type"])
        rows.append(ts)

turning_offsets = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
display(turning_offsets.head(20))

if not turning_offsets.empty:
    summary = turning_offsets.groupby("type").agg(
        reps=("rep", "count"),
        median_abs_max_offset=("max_offset", lambda s: float(np.nanmedian(np.abs(s)))),
        median_abs_min_offset=("min_offset", lambda s: float(np.nanmedian(np.abs(s)))),
        p90_abs_max_offset=("max_offset", lambda s: float(np.nanpercentile(np.abs(s), 90))),
        p90_abs_min_offset=("min_offset", lambda s: float(np.nanpercentile(np.abs(s), 90))),
    )
    display(summary)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    turning_offsets.to_csv(OUT_DIR / "turning_point_offsets.csv", index=False, encoding="utf-8-sig")
    print("Saved:", OUT_DIR / "turning_point_offsets.csv")

In [11]:
import cv2
import mediapipe as mp
from pathlib import Path

INPUT_VIDEO     = "./data/측면/data/benchpress/stu8_12.mp4"
OUTPUT_VIDEO    = "./infer/output/stu8_12output_pose.mp4"
TRIM_START_SEC = 12.0  # 앞에서 자를 초
TRIM_END_SEC   = 10.0  # 뒤에서 자를 초

output_path = Path(OUTPUT_VIDEO)
output_path.parent.mkdir(parents=True, exist_ok=True)

mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"영상 열기 실패: {INPUT_VIDEO}")

src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

skip_start = int(src_fps * TRIM_START_SEC)
skip_end   = int(src_fps * TRIM_END_SEC)
end_frame  = total - skip_end
remaining  = end_frame - skip_start

if remaining <= 0:
    raise RuntimeError(f"자를 구간이 영상 전체를 초과함: total={total}, start={skip_start}, end={skip_end}")

cap.set(cv2.CAP_PROP_POS_FRAMES, skip_start)
print(f"입력: {width}x{height} @ {src_fps:.2f}fps, 총 {total}프레임")
print(f"앞 {TRIM_START_SEC}초({skip_start}f) / 뒤 {TRIM_END_SEC}초({skip_end}f) 제거")
print(f"출력 프레임: {remaining}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(output_path), fourcc, src_fps, (width, height))
if not writer.isOpened():
    raise RuntimeError(f"VideoWriter 열기 실패: {output_path}")

with mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as pose:
    frame_i = 0
    while cap.isOpened():
        cur_pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if cur_pos >= end_frame:
            break

        ret, frame = cap.read()
        if not ret:
            break

        rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose.process(rgb)

        if result.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                result.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
            )

        writer.write(frame)
        frame_i += 1
        if frame_i % 30 == 0:
            print(f"{frame_i}/{remaining} ({frame_i/remaining*100:.1f}%)")

cap.release()
writer.release()

size_mb = output_path.stat().st_size / 1024 / 1024
print(f"[DONE] {output_path} | {frame_i}프레임 | {size_mb:.1f} MB")

입력: 640x360 @ 25.00fps, 총 900프레임
앞 12.0초(300f) / 뒤 10.0초(250f) 제거
출력 프레임: 350
30/350 (8.6%)
60/350 (17.1%)
90/350 (25.7%)
120/350 (34.3%)
150/350 (42.9%)
180/350 (51.4%)
210/350 (60.0%)
240/350 (68.6%)
270/350 (77.1%)
300/350 (85.7%)
330/350 (94.3%)
[DONE] infer\output\stu8_12output_pose.mp4 | 350프레임 | 6.5 MB
